In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score
from itertools import combinations

In [4]:
DROPPED = ['Class', 'Time', 'Amount', 'V23', 'V27', 'V28',
           'V26', 'V25', 'V22', 'V15', 'V13', 'V24']

# Strategy:
#   'exhaustive'  – try every combination (only feasible if n_features <= ~18)
#   'stepwise'    – forward stepwise selection (fast, greedy)
#   'top_n'       – exhaustive over only the top N features by importance
TOP_N    = 12        # used when STRATEGY == 'top_n'
MIN_FEATURES = 3     # minimum subset size to test
OPTIMIZE_FOR = 'recall'   # 'recall', 'precision', or 'f1'

In [5]:
df = pd.read_csv('creditcard.csv')
ALL_FEATURES = [c for c in df.columns if c not in DROPPED]
print(f"Available features ({len(ALL_FEATURES)}): {ALL_FEATURES}\n")

X_all = df[ALL_FEATURES]
y     = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=42
)

def evaluate(features):
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train[list(features)], y_train)
    y_pred = model.predict(X_test[list(features)])
    return {
        'recall':    recall_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
    }

results = []

Available features (19): ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V14', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21']



In [10]:
# Forward Stepwise
print("Forward stepwise selection…\n")
selected   = []
remaining  = list(ALL_FEATURES)

while remaining:
    best_score  = -1
    best_feat   = None
    for feat in remaining:
        candidate = selected + [feat]
        if len(candidate) < MIN_FEATURES:
            # still track score but don't record yet
            m = evaluate(candidate)
            score = m[OPTIMIZE_FOR]
        else:
            m = evaluate(candidate)
            score = m[OPTIMIZE_FOR]
            results.append({'features': list(candidate), **m})

        if score > best_score:
            best_score = score
            best_feat  = feat
            best_metrics = m

    selected.append(best_feat)
    remaining.remove(best_feat)
    print(f"  Added '{best_feat}' → "
            f"recall={best_metrics['recall']:.4f}  "
            f"precision={best_metrics['precision']:.4f}  "
            f"f1={best_metrics['f1']:.4f}  "
            f"[{len(selected)} features]")

Forward stepwise selection…

  Added 'V12' → recall=0.8776  precision=0.0216  f1=0.0422  [1 features]
  Added 'V14' → recall=0.8980  precision=0.0620  f1=0.1160  [2 features]
  Added 'V4' → recall=0.9286  precision=0.0510  f1=0.0968  [3 features]
  Added 'V1' → recall=0.9286  precision=0.0525  f1=0.0994  [4 features]
  Added 'V2' → recall=0.9286  precision=0.0520  f1=0.0985  [5 features]
  Added 'V3' → recall=0.9286  precision=0.0516  f1=0.0978  [6 features]
  Added 'V5' → recall=0.9286  precision=0.0500  f1=0.0949  [7 features]
  Added 'V6' → recall=0.9286  precision=0.0499  f1=0.0947  [8 features]
  Added 'V7' → recall=0.9286  precision=0.0490  f1=0.0931  [9 features]
  Added 'V8' → recall=0.9286  precision=0.0485  f1=0.0922  [10 features]
  Added 'V9' → recall=0.9286  precision=0.0484  f1=0.0921  [11 features]
  Added 'V10' → recall=0.9286  precision=0.0550  f1=0.1038  [12 features]
  Added 'V11' → recall=0.9286  precision=0.0563  f1=0.1061  [13 features]
  Added 'V17' → recall=0.92

In [12]:
# Top-N exhaustive
print("Ranking features by logistic regression importance…")
base_model = LogisticRegression(max_iter=1000, class_weight='balanced')
base_model.fit(X_train, y_train)
importances = pd.Series(np.abs(base_model.coef_[0]), index=ALL_FEATURES)
top_features = importances.nlargest(TOP_N).index.tolist()
print(f"Top {TOP_N} features: {top_features}\n")

total = sum(1 for k in range(MIN_FEATURES, len(top_features)+1)
            for _ in combinations(top_features, k))
print(f"Exhaustive search over top {TOP_N}: {total:,} combinations…\n")
done = 0
for k in range(MIN_FEATURES, len(top_features) + 1):
    for combo in combinations(top_features, k):
        metrics = evaluate(combo)
        results.append({'features': list(combo), **metrics})
        done += 1
        if done % 200 == 0:
            print(f"  {done:,} / {total:,} done…")

Ranking features by logistic regression importance…
Top 12 features: ['V4', 'V10', 'V16', 'V14', 'V12', 'V8', 'V17', 'V11', 'V20', 'V5', 'V21', 'V1']

Exhaustive search over top 12: 4,017 combinations…

  200 / 4,017 done…
  400 / 4,017 done…
  600 / 4,017 done…
  800 / 4,017 done…
  1,000 / 4,017 done…
  1,200 / 4,017 done…
  1,400 / 4,017 done…
  1,600 / 4,017 done…
  1,800 / 4,017 done…
  2,000 / 4,017 done…
  2,200 / 4,017 done…
  2,400 / 4,017 done…
  2,600 / 4,017 done…
  2,800 / 4,017 done…
  3,000 / 4,017 done…
  3,200 / 4,017 done…
  3,400 / 4,017 done…
  3,600 / 4,017 done…
  3,800 / 4,017 done…
  4,000 / 4,017 done…


In [13]:
results_df = pd.DataFrame(results)
results_df['n_features'] = results_df['features'].apply(len)
results_df = results_df.sort_values(OPTIMIZE_FOR, ascending=False).reset_index(drop=True)

print("\n══════════════════════════════════════════════════════")
print(f"TOP 10 COMBINATIONS BY {OPTIMIZE_FOR.upper()}")
print("══════════════════════════════════════════════════════")
for i, row in results_df.head(10).iterrows():
    print(f"\nRank {i+1}  ({row['n_features']} features)")
    print(f"  Recall:    {row['recall']:.4f}")
    print(f"  Precision: {row['precision']:.4f}")
    print(f"  F1:        {row['f1']:.4f}")
    print(f"  Features:  {row['features']}")

# Save full results
out_path = 'feature_search_results2.csv'
results_df['features'] = results_df['features'].apply(str)
results_df.to_csv(out_path, index=False)
print(f"\nAll results saved to '{out_path}'")


══════════════════════════════════════════════════════
TOP 10 COMBINATIONS BY RECALL
══════════════════════════════════════════════════════

Rank 1  (7 features)
  Recall:    0.9286
  Precision: 0.0565
  F1:        0.1064
  Features:  ['V4', 'V14', 'V12', 'V8', 'V17', 'V11', 'V1']

Rank 2  (7 features)
  Recall:    0.9286
  Precision: 0.0508
  F1:        0.0964
  Features:  ['V12', 'V14', 'V4', 'V1', 'V2', 'V3', 'V18']

Rank 3  (7 features)
  Recall:    0.9286
  Precision: 0.0516
  F1:        0.0978
  Features:  ['V12', 'V14', 'V4', 'V1', 'V2', 'V3', 'V20']

Rank 4  (7 features)
  Recall:    0.9286
  Precision: 0.0513
  F1:        0.0972
  Features:  ['V12', 'V14', 'V4', 'V1', 'V2', 'V3', 'V21']

Rank 5  (8 features)
  Recall:    0.9286
  Precision: 0.0499
  F1:        0.0947
  Features:  ['V12', 'V14', 'V4', 'V1', 'V2', 'V3', 'V5', 'V6']

Rank 6  (8 features)
  Recall:    0.9286
  Precision: 0.0489
  F1:        0.0930
  Features:  ['V12', 'V14', 'V4', 'V1', 'V2', 'V3', 'V5', 'V7']

R

In [16]:
forward_stepwise = pd.read_csv('forward_stepwise_logreg.csv')
top_n = pd.read_csv('top_n_logreg.csv')

BEST_OF = 10


In [17]:
fs_sorted = forward_stepwise.sort_values("recall", ascending=False).reset_index(drop=True)
top_sorted = top_n.sort_values("recall", ascending=False).reset_index(drop=True)

print("\n══════════════════════════════════════════════════════")
print("BEST OF EACH — SIDE BY SIDE")
print("══════════════════════════════════════════════════════")
metrics = ['recall', 'precision', 'f1']
summary = pd.DataFrame({
    'Metric': metrics,
    'File1_best': [fs_sorted[m].iloc[0] for m in metrics],
    'File2_best': [top_sorted[m].iloc[0] for m in metrics],
})
summary['Difference'] = summary['File2_best'] - summary['File1_best']
summary['Winner'] = summary['Difference'].apply(lambda x: 'File2' if x > 0 else ('File1' if x < 0 else 'Tie'))
print(summary.to_string(index=False))



══════════════════════════════════════════════════════
BEST OF EACH — SIDE BY SIDE
══════════════════════════════════════════════════════
   Metric  File1_best  File2_best  Difference Winner
   recall    0.928571    0.928571    0.000000    Tie
precision    0.058559    0.056452   -0.002107  File1
       f1    0.110169    0.106433   -0.003737  File1


In [14]:
fs_sorted = forward_stepwise.sort_values("f1", ascending=False).reset_index(drop=True)
top_sorted = top_n.sort_values("f1", ascending=False).reset_index(drop=True)

print("\n══════════════════════════════════════════════════════")
print("BEST OF EACH — SIDE BY SIDE")
print("══════════════════════════════════════════════════════")
metrics = ['recall', 'precision', 'f1']
summary = pd.DataFrame({
    'Metric': metrics,
    'File1_best': [fs_sorted[m].iloc[0] for m in metrics],
    'File2_best': [top_sorted[m].iloc[0] for m in metrics],
})
summary['Difference'] = summary['File2_best'] - summary['File1_best']
summary['Winner'] = summary['Difference'].apply(lambda x: 'File2' if x > 0 else ('File1' if x < 0 else 'Tie'))
print(summary.to_string(index=False))


══════════════════════════════════════════════════════
BEST OF EACH — SIDE BY SIDE
══════════════════════════════════════════════════════
   Metric  File1_best  File2_best  Difference Winner
   recall    0.897959    0.775510   -0.122449  File1
precision    0.068270    0.143396    0.075126  File2
       f1    0.126893    0.242038    0.115146  File2


In [15]:
fs_sorted = forward_stepwise.sort_values("precision", ascending=False).reset_index(drop=True)
top_sorted = top_n.sort_values("precision", ascending=False).reset_index(drop=True)

print("\n══════════════════════════════════════════════════════")
print("BEST OF EACH — SIDE BY SIDE")
print("══════════════════════════════════════════════════════")
metrics = ['recall', 'precision', 'f1']
summary = pd.DataFrame({
    'Metric': metrics,
    'File1_best': [fs_sorted[m].iloc[0] for m in metrics],
    'File2_best': [top_sorted[m].iloc[0] for m in metrics],
})
summary['Difference'] = summary['File2_best'] - summary['File1_best']
summary['Winner'] = summary['Difference'].apply(lambda x: 'File2' if x > 0 else ('File1' if x < 0 else 'Tie'))
print(summary.to_string(index=False))


══════════════════════════════════════════════════════
BEST OF EACH — SIDE BY SIDE
══════════════════════════════════════════════════════
   Metric  File1_best  File2_best  Difference Winner
   recall    0.897959    0.775510   -0.122449  File1
precision    0.068270    0.143396    0.075126  File2
       f1    0.126893    0.242038    0.115146  File2
